# Attention optimization: compute and memory trade-offs

Prerequisites: scaled dot-product attention and causal masks. Estimated time: 30 minutes. Objectives: compare full and local attention score work, measure toy approximation error, and identify when exact fused kernels are preferable.

This dependency-free toy compares exact full causal attention with a local-window approximation. It makes the quality-versus-compute trade-off visible; it does not benchmark fused GPU kernels such as FlashAttention.

## Assumptions and baseline

Use scalar query/key/value vectors and stable softmax. Full causal attention at position i considers positions 0 through i. The number of score pairs is n(n+1)/2.

In [ ]:
from math import exp
queries = [1.0, 0.5, -0.5, 1.5]
keys = [0.0, 1.0, 2.0, -1.0]
values = [2.0, 0.0, 1.0, 4.0]
def attend(q, end, start=0):
    scores = [q * keys[j] for j in range(start, end + 1)]
    peak = max(scores)
    weights = [exp(s - peak) for s in scores]
    total = sum(weights)
    return sum(w * values[start + i] for i, w in enumerate(weights)) / total
full = [attend(queries[i], i) for i in range(len(queries))]
full_pairs = len(queries) * (len(queries) + 1) // 2
assert len(full) == 4 and full_pairs == 10
print('full causal outputs:', [round(x, 4) for x in full], 'score pairs:', full_pairs)

## Local attention alternative

A window of two retains at most the current and immediately previous token. This reduces pair computation, but it changes the receptive field and may omit useful distant context. A fused exact kernel can reduce memory traffic without making this approximation.

In [ ]:
window = 2
local = [attend(queries[i], i, max(0, i - window + 1)) for i in range(len(queries))]
local_pairs = sum(min(i + 1, window) for i in range(len(queries)))
assert local_pairs == 7 and local_pairs < full_pairs
print('local outputs:', [round(x, 4) for x in local], 'score pairs:', local_pairs)

## Comparison and metric

Pair count falls from ten to seven (30%). Measure approximation error as mean absolute output difference on this toy. This is not a task-quality metric; real window selection needs long-range task tests. Exact tiled attention should instead match full attention within numerical tolerance while reducing peak score-matrix storage.

In [ ]:
mae = sum(abs(a - b) for a, b in zip(full, local)) / len(full)
assert 0 < mae < 1
assert round((full_pairs - local_pairs) / full_pairs, 2) == 0.30
print('pair reduction:', round(100 * (full_pairs-local_pairs) / full_pairs, 1), '%; toy output MAE:', round(mae, 4))

## Edge and failure cases

A window of one is self-only and discards all prior context. A window larger than the sequence should recover full causal attention. Softmax must remain stable for large scores; subtracting the maximum prevents overflow.

In [ ]:
self_only = [attend(queries[i], i, i) for i in range(len(queries))]
wide = [attend(queries[i], i, max(0, i - 99)) for i in range(len(queries))]
assert self_only == values
assert all(abs(a-b) < 1e-12 for a, b in zip(full, wide))
# Stable normalization remains finite for very large logits.
assert attend(1000.0, 3) == values[2]

## Exercises

1. Try windows 1, 2, 3, and 4; plot or tabulate pair count and MAE.
2. Design an exact blockwise algorithm that never materializes the full score matrix, then state its numerical comparison test.
3. Explain why fewer arithmetic operations do not guarantee faster wall-clock execution on every accelerator.

## Summary

Attention optimization has distinct paths: exact kernel/memory improvements preserve semantics, while sparse or local attention can reduce work by changing which tokens interact. Report both resource metrics and quality impact. See the [concept note](../concepts/24-attention-optimization.md) and the [practice assessment](../interview-prep/curriculum-practice.md#attention-optimization).